# Signal Protocol with Lattice-based Station-to-Station (LatticeSTS)

This notebook implements a Lattice-based version of the Station-to-Station (STS) protocol, designed to replace the long-term identity key exchange in the Signal Protocol. LatticeSTS provides post-quantum authentication and confidentiality for the root key establishment phase.

In [1]:
import numpy as np
import time
import hashlib
import math
import os

## 1. LatticeSTS Implementation

We implement the core primitives for Lattice-based key exchange based on the SIS (Short Integer Solution) problem over a ring.

In [2]:
class LatticeSTS:
    def __init__(self, n=256):
        self.n = n
        self.q = 2 * (n**2) + 1
        self.m = 2 * n
        self.beta = int(np.sqrt(self.m))
        # Shared public matrix A (can be derived from a seed)
        np.random.seed(123) # For reproducibility
        self.A = np.random.randint(0, self.q, (self.m, self.m))
        
    def generate_keypair_alice(self):
        # Alice secret key is a short vector s_A
        s_A = np.random.randint(-self.beta//2, self.beta//2, self.m)
        # Public key p_A = A * s_A mod q
        p_A = (self.A @ s_A) % self.q
        return s_A, p_A
        
    def generate_keypair_bob(self):
        # Bob secret key is a short vector s_B
        s_B = np.random.randint(-self.beta//2, self.beta//2, self.m)
        # Public key p_B = s_B * A mod q
        p_B = (s_B @ self.A) % self.q
        return s_B, p_B
        
    def compute_shared_secret_alice(self, s_A, p_B):
        # val = p_B * s_A mod q = (s_B * A) * s_A mod q
        val = (p_B @ s_A) % self.q
        num_bytes = math.ceil(self.q.bit_length() / 8)
        return hashlib.sha256(int(val).to_bytes(num_bytes, 'big')).digest()

    def compute_shared_secret_bob(self, s_B, p_A):
        # val = s_B * p_A mod q = s_B * (A * s_A) mod q
        val = (s_B @ p_A) % self.q
        num_bytes = math.ceil(self.q.bit_length() / 8)
        return hashlib.sha256(int(val).to_bytes(num_bytes, 'big')).digest()

## 2. Handshake Simulation

We simulate the long-term key exchange between Alice and Bob.

In [3]:
sts = LatticeSTS()

print("1. Alice and Bob generate their long-term identity keys.")
s_A, p_A = sts.generate_keypair_alice()
s_B, p_B = sts.generate_keypair_bob()

print("2. They exchange public keys and compute the shared master secret.")
ms_alice = sts.compute_shared_secret_alice(s_A, p_B)
ms_bob = sts.compute_shared_secret_bob(s_B, p_A)

print(f"Alice's Master Secret: {ms_alice.hex()}")
print(f"Bob's Master Secret:   {ms_bob.hex()}")
assert ms_alice == ms_bob, "Master secret mismatch!"

1. Alice and Bob generate their long-term identity keys.
2. They exchange public keys and compute the shared master secret.
Alice's Master Secret: 84a6ee8d7c0153e51377127539ec3331e830c4dbb8503b99459694a925bc01d4
Bob's Master Secret:   84a6ee8d7c0153e51377127539ec3331e830c4dbb8503b99459694a925bc01d4


## 3. Metric Analysis

In [4]:
def analyze_metrics():
    iterations = 100
    total_gen_alice = 0
    total_gen_bob = 0
    total_compute = 0
    
    for _ in range(iterations):
        start = time.time()
        s_A, p_A = sts.generate_keypair_alice()
        total_gen_alice += (time.time() - start)
        
        start = time.time()
        s_B, p_B = sts.generate_keypair_bob()
        total_gen_bob += (time.time() - start)
        
        start = time.time()
        sts.compute_shared_secret_alice(s_A, p_B)
        total_compute += (time.time() - start)
        
    print(f"--- Average Execution Times ({iterations} iterations) ---")
    print(f"Alice Key Gen:  {total_gen_alice / iterations * 1000:.3f} ms")
    print(f"Bob Key Gen:    {total_gen_bob / iterations * 1000:.3f} ms")
    print(f"Secret Compute: {total_compute / iterations * 1000:.3f} ms")
    
    print("\n--- Key Sizes ---")
    print(f"Public Key (p_A/p_B): {p_A.nbytes} bytes")
    print(f"Secret Key (s_A/s_B): {s_A.nbytes} bytes")
    print(f"Master Secret:        {len(ms_alice)} bytes")

analyze_metrics()

--- Average Execution Times (100 iterations) ---
Alice Key Gen:  0.173 ms
Bob Key Gen:    0.541 ms
Secret Compute: 0.008 ms

--- Key Sizes ---
Public Key (p_A/p_B): 4096 bytes
Secret Key (s_A/s_B): 4096 bytes
Master Secret:        32 bytes


## 4. Security Analysis

### Authentication and Confidentiality
LatticeSTS ensures that only the holders of the secret keys $\mathbf{s}_A$ and $\mathbf{s}_B$ can compute the shared master secret. This provides mutual authentication in the root key establishment phase. 

### Quantum Resistance
The security of LatticeSTS rests on the difficulty of finding a short solution to the Inhomogeneous SIS problem, which is known to be hard for both classical and quantum computers. By using LatticeSTS for identity keys, the protocol ensures that even if an attacker archives current handshakes, they cannot decrypt them in the future using a quantum computer to break the long-term identities.